In [2]:
import pandas as pd
import os
import re

import spacy

from sklearn.feature_extraction.text import CountVectorizer

In [3]:
os.getcwd()

'd:\\Yilan\\1\\Research Projects\\steamreviews_moral\\analysis'

In [4]:
os.makedirs("data_conditions", exist_ok=True)

In [6]:
#Lable games based on IVs

game_conditions = {
    "282070": {"MD": "Low",  "Agency": "High"},  # This War of Mine
    "1227530": {"MD": "Low",  "Agency": "High"}, # Partisans 1941
    "15390":  {"MD": "Low",  "Agency": "Low"},  # Brothers in Arms
    "50300":  {"MD": "Low",  "Agency": "Low"},  # Spec Ops
    "287700": {"MD": "High", "Agency": "High"}, # MGSV
    "460930": {"MD": "High", "Agency": "High"}, # Wildlands
    "400750": {"MD": "High", "Agency": "Low"},  # Gates of Hell
    "1029690":{"MD": "High", "Agency": "Low"}   # Sniper Elite 5
}

In [7]:
# Create a full dataset for all games

data_dir = "data"

dfs = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        app_id = file.replace(".csv", "")
        path = os.path.join(data_dir, file)

        df = pd.read_csv(path)

        if app_id in game_conditions:
            df["app_id"] = app_id
            df["MD"] = game_conditions[app_id]["MD"]
            df["Agency"] = game_conditions[app_id]["Agency"]

            dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True) 

print("Total samples:", len(full_df))

Total samples: 49815


## Condition-Based Subsets
We created multiple subsets of the data:

- By Moral Disengagement:
  - MD High
  - MD Low

- By Player Agency:
  - Agency High
  - Agency Low

- By combined conditions:
  - HH (High MD × High Agency)
  - HL (High MD × Low Agency)
  - LH (Low MD × High Agency)
  - LL (Low MD × Low Agency)

In [8]:
#Moral disengagement (High vs Low)
df_MD_H = full_df[full_df["MD"] == "High"]
df_MD_L = full_df[full_df["MD"] == "Low"]
print(len(df_MD_H), len(df_MD_L))

#Player agency (High vs Low)
df_Agency_H = full_df[full_df["Agency"] == "High"]
df_Agency_L = full_df[full_df["Agency"] == "Low"]
print(len(df_Agency_H), len(df_Agency_L))

# 2 x 2 conditions
df_HH = full_df[(full_df["MD"] == "High") & (full_df["Agency"] == "High")]
df_HL = full_df[(full_df["MD"] == "High") & (full_df["Agency"] == "Low")]
df_LH = full_df[(full_df["MD"] == "Low")  & (full_df["Agency"] == "High")]
df_LL = full_df[(full_df["MD"] == "Low")  & (full_df["Agency"] == "Low")]

print("HH:", len(df_HH))
print("HL:", len(df_HL))
print("LH:", len(df_LH))
print("LL:", len(df_LL))

19260 30555
34195 15620
HH: 14195
HL: 5065
LH: 20000
LL: 10555


In [45]:
# Save as csv files

output_dir = "data_conditions"

full_df.to_csv(os.path.join(output_dir, "full.csv"), index=False)

df_MD_H.to_csv(os.path.join(output_dir, "MD_H.csv"), index=False)
df_MD_L.to_csv(os.path.join(output_dir, "MD_L.csv"), index=False)

df_Agency_H.to_csv(os.path.join(output_dir, "Agency_H.csv"), index=False)
df_Agency_L.to_csv(os.path.join(output_dir, "Agency_L.csv"), index=False)

df_HH.to_csv(os.path.join(output_dir, "HH.csv"), index=False)
df_HL.to_csv(os.path.join(output_dir, "HL.csv"), index=False)
df_LH.to_csv(os.path.join(output_dir, "LH.csv"), index=False)
df_LL.to_csv(os.path.join(output_dir, "LL.csv"), index=False)

In [9]:
# Tokenization

def preprocess_texts(texts):
    processed = []

    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [
            token.lemma_
            for token in doc
            if token.is_alpha and not token.is_stop and len(token) > 2
        ]
        processed.append(" ".join(tokens))

    return processed

In [41]:
clean_full = preprocess_texts(full_df["review_text"].dropna())
clean_texts[:5]

['people write bug know maybe lucky catch couple entire game cure load quick save game bad entertaining play knife mine minimal shoot mission necessary',
 'hard play stealth game fun game mess learn yeah cool game',
 'gameplay good exceptional compare game genre base mechanic fun recommend manage hour game story dialogue german evil Germans die torture poison mutilate Germans great freedom',
 'buggy little well okay basecamp phase bit uninteresting feel like cut thing little bit underbaked worth price',
 'sale complete waste money yeah game buggy super buggy think game annoying bug wise dumb dialog kid play army scenario waste money think change fun wrong glad buy dlc']

In [42]:
full_df_clean = full_df.dropna(subset=["review_text"]).copy()
full_df_clean["clean_text"] = clean_full

In [43]:
df_MD_H_clean = full_df_clean[full_df_clean["MD"] == "High"]
df_MD_L_clean = full_df_clean[full_df_clean["MD"] == "Low"]

df_HH_clean = full_df_clean[(full_df_clean["MD"] == "High") & (full_df_clean["Agency"] == "High")]
df_HL_clean = full_df_clean[(full_df_clean["MD"] == "High") & (full_df_clean["Agency"] == "Low")]
df_LH_clean = full_df_clean[(full_df_clean["MD"] == "Low") & (full_df_clean["Agency"] == "High")]
df_LL_clean = full_df_clean[(full_df_clean["MD"] == "Low") & (full_df_clean["Agency"] == "Low")]